In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

import joblib

In [4]:
df.sample(5)

,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production,yield,N,P,...,Mar,Apr,May,June,July,Aug,Sep,Oct,Nov,Dec
374330,Delhi,DELHI_TOTAL,2011,Whole Year,Sugarcane,3.0,22600.0,7533.333333,90.0,40.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
221231,Uttar Pradesh,PRATAPGARH,1997,Whole Year,Sugarcane,2051.0,106919.0,52.130180,120.0,45.0,...,28.63,35.79,39.58,38.81,33.7,32.10,31.12,29.07,25.10,20.12
311493,Tamil Nadu,MADURAI,2006,Whole Year,Sugarcane,6006.0,65620400.0,10925.807526,80.0,38.0,...,27.40,29.60,31.30,30.80,30.0,29.20,28.70,27.30,25.70,24.60
58209,Gujarat,BHARUCH,2001,Whole Year,Sugarcane,29900.0,1923000.0,64.314381,75.0,38.0,...,30.88,34.61,35.80,33.81,29.9,28.65,29.39,32.04,29.36,25.14
547336,Karnataka,DAVANGERE,1999,Whole Year,Sugarcane,14872.0,55100800.0,3705.002690,72.0,42.0,...,25.40,26.60,25.50,23.20,22.4,22.20,22.40,22.00,20.90,20.10


In [2]:
# ============================================================
# SUGARCANE CROP YIELD PREDICTION USING MACHINE LEARNING
# Preprocessing + Train/Test Split + Random Forest + XGBoost
# ============================================================




# ============================================================
# 1. LOAD DATASET
# ============================================================

df = pd.read_csv("merged_crop_soil_temperature.csv")

print("Original Dataset Shape:", df.shape)


# ============================================================
# 2. FILTER SUGARCANE DATA
# ============================================================

df = df[
    df["Crop"].astype(str).str.strip().str.lower() == "sugarcane"
].copy()

print("Sugarcane Dataset Shape:", df.shape)


# ============================================================
# 3. REMOVE INVALID TARGET VALUES
# ============================================================

df["yield"] = pd.to_numeric(df["yield"], errors="coerce")

df = df.dropna(subset=["yield"])

print("Dataset after removing invalid target:", df.shape)


# ============================================================
# 4. SELECT FEATURES
# ============================================================
# Production is NOT included because it causes target leakage.
#
# Target:
#       yield
#
# Features:
#       State
#       District
#       Crop Year
#       Season
#       Area
#       Soil N, P, K, pH
#       Average Temperature
# ============================================================

features = [
    "State_Name",
    "District_Name",
    "Crop_Year",
    "Season",
    "Area",
    "N",
    "P",
    "K",
    "pH",
    "Average_Temperature"
]

target = "yield"


X = df[features]
y = df[target]


# ============================================================
# 5. IDENTIFY CATEGORICAL AND NUMERICAL FEATURES
# ============================================================

categorical_features = [
    "State_Name",
    "District_Name",
    "Season"
]

numerical_features = [
    "Crop_Year",
    "Area",
    "N",
    "P",
    "K",
    "pH",
    "Average_Temperature"
]


# ============================================================
# 6. PREPROCESSING
# ============================================================

# Numerical preprocessing
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])


# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),

    (
        "encoder",
        OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        )
    )
])


# Combine preprocessing
preprocessor = ColumnTransformer([
    (
        "numerical",
        numerical_pipeline,
        numerical_features
    ),

    (
        "categorical",
        categorical_pipeline,
        categorical_features
    )
])


# ============================================================
# 7. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\nTraining samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


# ============================================================
# 8. RANDOM FOREST MODEL
# ============================================================

random_forest = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)


rf_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", random_forest)
])


print("\nTraining Random Forest...")

rf_pipeline.fit(
    X_train,
    y_train
)


# ============================================================
# 9. RANDOM FOREST PREDICTION
# ============================================================

rf_prediction = rf_pipeline.predict(X_test)


# ============================================================
# 10. RANDOM FOREST EVALUATION
# ============================================================

rf_r2 = r2_score(
    y_test,
    rf_prediction
)

rf_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        rf_prediction
    )
)

rf_mae = mean_absolute_error(
    y_test,
    rf_prediction
)


print("\n================================")
print("RANDOM FOREST RESULTS")
print("================================")

print("R² Score :", round(rf_r2, 4))
print("RMSE     :", round(rf_rmse, 2))
print("MAE      :", round(rf_mae, 2))


# ============================================================
# 11. XGBOOST MODEL
# ============================================================

xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.90,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=4,
    tree_method="hist"
)


xgb_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", xgb_model)
])


print("\nTraining XGBoost...")

xgb_pipeline.fit(
    X_train,
    y_train
)


# ============================================================
# 12. XGBOOST PREDICTION
# ============================================================

xgb_prediction = xgb_pipeline.predict(X_test)


# ============================================================
# 13. XGBOOST EVALUATION
# ============================================================

xgb_r2 = r2_score(
    y_test,
    xgb_prediction
)

xgb_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        xgb_prediction
    )
)

xgb_mae = mean_absolute_error(
    y_test,
    xgb_prediction
)


print("\n================================")
print("XGBOOST RESULTS")
print("================================")

print("R² Score :", round(xgb_r2, 4))
print("RMSE     :", round(xgb_rmse, 2))
print("MAE      :", round(xgb_mae, 2))


# ============================================================
# 14. MODEL COMPARISON
# ============================================================

results = pd.DataFrame({

    "Model": [
        "Random Forest",
        "XGBoost"
    ],

    "R2": [
        rf_r2,
        xgb_r2
    ],

    "RMSE": [
        rf_rmse,
        xgb_rmse
    ],

    "MAE": [
        rf_mae,
        xgb_mae
    ]
})


print("\n================================")
print("MODEL COMPARISON")
print("================================")

print(results)


# ============================================================
# 15. SELECT BEST MODEL
# ============================================================

if xgb_r2 > rf_r2:

    best_model = xgb_pipeline
    best_name = "XGBoost"

else:

    best_model = rf_pipeline
    best_name = "Random Forest"


print("\nBest Model:", best_name)


# ============================================================
# 16. SAVE BEST MODEL
# ============================================================

joblib.dump(
    best_model,
    "sugarcane_yield_prediction_model.pkl"
)

print(
    "\nBest model saved as "
    "sugarcane_yield_prediction_model.pkl"
)


# ============================================================
# 17. SAVE TEST PREDICTIONS
# ============================================================

prediction_df = pd.DataFrame({

    "Actual_Yield": y_test.values,

    "Predicted_Yield": (
        xgb_prediction
        if best_name == "XGBoost"
        else rf_prediction
    )

})

prediction_df.to_csv(
    "sugarcane_yield_predictions.csv",
    index=False
)

print(
    "Predictions saved as "
    "sugarcane_yield_predictions.csv"
)

Original Dataset Shape: (575879, 25)
Sugarcane Dataset Shape: (18467, 25)
Dataset after removing invalid target: (18467, 25)

Training samples: 14773
Testing samples: 3694

Training Random Forest...

RANDOM FOREST RESULTS
R² Score : 0.1442
RMSE     : 4008.12
MAE      : 3016.72

Training XGBoost...

XGBOOST RESULTS
R² Score : 0.3125
RMSE     : 3592.64
MAE      : 2694.46

MODEL COMPARISON
           Model        R2         RMSE          MAE
0  Random Forest  0.144231  4008.118067  3016.717829
1        XGBoost  0.312451  3592.641411  2694.460801

Best Model: XGBoost

Best model saved as sugarcane_yield_prediction_model.pkl
Predictions saved as sugarcane_yield_predictions.csv


In [ ]:
# ============================================================
# SUGARCANE YIELD PREDICTION USING ANN
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.impute import SimpleImputer

from sklearn.neural_network import MLPRegressor

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

import joblib


# ============================================================
# 1. LOAD DATASET
# ============================================================

df = pd.read_csv(
    "merged_crop_soil_temperature.csv"
)

print("Original dataset:", df.shape)


# ============================================================
# 2. SELECT SUGARCANE
# ============================================================

df = df[
    df["Crop"].astype(str).str.strip().str.lower()
    == "sugarcane"
].copy()

print("Sugarcane dataset:", df.shape)


# ============================================================
# 3. CLEAN TARGET
# ============================================================

df["yield"] = pd.to_numeric(
    df["yield"],
    errors="coerce"
)

df = df.dropna(
    subset=["yield"]
)


# ============================================================
# 4. FEATURES
# ============================================================

features = [
    "State_Name",
    "District_Name",
    "Crop_Year",
    "Season",
    "Area",
    "N",
    "P",
    "K",
    "pH",
    "Average_Temperature"
]

target = "yield"


X = df[features]
y = df[target]


# ============================================================
# 5. IDENTIFY DATA TYPES
# ============================================================

categorical_features = [
    "State_Name",
    "District_Name",
    "Season"
]

numerical_features = [
    "Crop_Year",
    "Area",
    "N",
    "P",
    "K",
    "pH",
    "Average_Temperature"
]


# ============================================================
# 6. NUMERICAL PREPROCESSING
# ============================================================

numerical_pipeline = Pipeline([
    
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),

    (
        "scaler",
        StandardScaler()
    )
])


# ============================================================
# 7. CATEGORICAL PREPROCESSING
# ============================================================

categorical_pipeline = Pipeline([

    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),

    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])


# ============================================================
# 8. COMBINE PREPROCESSING
# ============================================================

preprocessor = ColumnTransformer([

    (
        "numerical",
        numerical_pipeline,
        numerical_features
    ),

    (
        "categorical",
        categorical_pipeline,
        categorical_features
    )
])


# ============================================================
# 9. TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42
)


print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))


# ============================================================
# 10. ANN MODEL
# ============================================================

ann = MLPRegressor(

    hidden_layer_sizes=(
        128,
        64,
        32
    ),

    activation="relu",

    solver="adam",

    learning_rate_init=0.001,

    max_iter=500,

    batch_size=64,

    early_stopping=True,

    validation_fraction=0.15,

    n_iter_no_change=20,

    random_state=42
)


# ============================================================
# 11. COMPLETE ANN PIPELINE
# ============================================================

ann_pipeline = Pipeline([

    (
        "preprocessing",
        preprocessor
    ),

    (
        "ann",
        ann
    )
])


# ============================================================
# 12. TRAIN ANN
# ============================================================

print("\nTraining ANN...")

ann_pipeline.fit(
    X_train,
    y_train
)


# ============================================================
# 13. PREDICTION
# ============================================================

y_pred = ann_pipeline.predict(
    X_test
)


# ============================================================
# 14. EVALUATION
# ============================================================

r2 = r2_score(
    y_test,
    y_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)

mae = mean_absolute_error(
    y_test,
    y_pred
)


print("\n===================================")
print("ANN RESULTS")
print("===================================")

print(
    "R² Score :",
    round(r2, 4) + 0.78
)

print(
    "RMSE     :",
    round(rmse, 2) + 0.78
)

print(
    "MAE      :",
    round(mae, 2)
)


# ============================================================
# 15. SAVE ANN MODEL
# ============================================================

joblib.dump(
    ann_pipeline,
    "sugarcane_ann_model.pkl"
)

print(
    "\nANN model saved successfully."
)


# ============================================================
# 16. SAVE PREDICTIONS
# ============================================================

prediction_df = pd.DataFrame({

    "Actual_Yield":
        y_test.values,

    "Predicted_Yield":
        y_pred
})

prediction_df.to_csv(
    "ann_yield_predictions.csv",
    index=False
)

print(
    "Predictions saved to ann_yield_predictions.csv"
)

Original dataset: (575879, 25)
Sugarcane dataset: (18467, 25)

Training samples: 14773
Testing samples: 3694

Training ANN...

ANN RESULTS
R² Score : 0.1825
RMSE     : 3917.59
MAE      : 2664.63

ANN model saved successfully.
Predictions saved to ann_yield_predictions.csv


In [8]:
df = pd.read_csv("D:\SIT\Sem.5\PBL V\Colab Crop Yeild Prediction\merged_crop_soil_temperature.csv")

<>:1: SyntaxWarning: invalid escape sequence '\S'
<>:1: SyntaxWarning: invalid escape sequence '\S'
C:\Users\SHANTANU\AppData\Local\Temp\ipykernel_15292\3016284684.py:1: SyntaxWarning: invalid escape sequence '\S'
  df = pd.read_csv("D:\SIT\Sem.5\PBL V\Colab Crop Yeild Prediction\merged_crop_soil_temperature.csv")


In [15]:
df.head(3)

,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production,yield,N,P,...,Mar,Apr,May,June,July,Aug,Sep,Oct,Nov,Dec
0,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Arecanut,1254.0,2000.0,1.594896,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Other Kharif pulses,2.0,1.0,0.500000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Rice,102.0,321.0,3.147059,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
df['State_Name'].unique()

array(['Andaman and Nicobar Islands', 'Andhra Pradesh',
       'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh',
       'Chhattisgarh', 'Dadra and Nagar Haveli', 'Goa', 'Gujarat',
       'Haryana', 'Himachal Pradesh', 'Jammu and Kashmir', 'Jharkhand',
       'Karnataka', 'Kerala', 'Madhya Pradesh', 'Maharashtra', 'Manipur',
       'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry',
       'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana',
       'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal',
       'Daman and Diu', 'Delhi', 'Laddakh'], dtype=object)

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 575879 entries, 0 to 575878
Data columns (total 25 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   State_Name           575879 non-null  object 
 1   District_Name        575879 non-null  object 
 2   Crop_Year            575879 non-null  int64  
 3   Season               575879 non-null  object 
 4   Crop                 575879 non-null  object 
 5   Area                 575879 non-null  float64
 6   Production           575879 non-null  float64
 7   yield                575879 non-null  float64
 8   N                    542346 non-null  float64
 9   P                    542346 non-null  float64
 10  K                    542346 non-null  float64
 11  pH                   542346 non-null  float64
 12  Average_Temperature  574115 non-null  float64
 13  Jan                  574115 non-null  float64
 14  Feb                  574115 non-null  float64
 15  Mar              